### Simple GenAI App Using LangChain And OpenAI Model

In [ ]:
# Import Libraries and Load all Environment Variables
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY") 
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY") 
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

In [ ]:
# Data Ingestion - Scrape the data from a website
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://docs.langchain.com/langsmith/observability-llm-tutorial")
docs = loader.load()
docs

In [ ]:
# Data Transformation - Converting Documents into Chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
documents = text_splitter.split_documents(docs)
documents

In [ ]:
# Embeddings - Converting Chunks into Vectors
from langchain_openai.embeddings import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

In [ ]:
# VectorStoreDB - Storing Vectors into VectoreStoreDB
from langchain_community.vectorstores import FAISS
vector_db = FAISS.from_documents(documents, embeddings)

In [ ]:
# Query from VectorStoreDB
query = "Once your app is working well in prototyping, you release it to a small group of real users."
result = vector_db.similarity_search_with_score(query)
result[0].page_content

In [ ]:
# Convert VectorStoreDB into Retriever
retriever = vector_db.as_retriever()

In [ ]:
# Load OpenAI LLM Model
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model = 'gpt-4o')
print(llm)

In [ ]:
# Create Custom Prompt
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    '''
Answer the following questions based on the provided context:
<context>
{context}
</context>

Question: {input}
'''
)

In [ ]:
# Create Document Chain
from langchain.chains.combine_documents import create_stuff_documents_chain

document_chain = create_stuff_documents_chain(llm, prompt)
document_chain

In [ ]:
# Create Retrieval Chain
from langchain.chains import create_retrieval_chain
retrieval_chain = create_retrieval_chain(retriever, document_chain)

In [ ]:
# Invoke Retrieval Chain
response = retrieval_chain.invoke({"input": "Once your app is working well in prototyping, you release it to a small group of real users."})
response["answer"]

##### Langchain Document and Retrieval Chain have Depreciated and Replaced With Langchain Expression Language (LCEL)